# Training a GNN on RelBench

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/snap-stanford/relbench/blob/relbench-hf/tutorials/gnn.ipynb)

A Graph Neural Network baseline for a RelBench entity task, with PyTorch Geometric
(graph) and PyTorch Frame (tabular features).

> **Use a GPU.** This notebook needs `torch` and `torch_geometric` and is best on a GPU.
> In Colab, switch the runtime to GPU (*Runtime → Change runtime type → GPU*) before running.

In [ ]:
# Install RelBench with the full (GNN) extras (skip if already installed locally)
!pip install relbench[full]

In [ ]:
import torch
import torch.nn.functional as F
from torch_frame.config.text_embedder import TextEmbedderConfig
from torch_frame.testing.text_embedder import HashTextEmbedder
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import MLP

import relbench
from relbench.modeling.graph import (
    get_node_train_table_input,
    make_pkey_fkey_graph,
)
from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE
from relbench.modeling.utils import get_stype_proposal

## Load the dataset and a (binary) entity task

In [ ]:
dataset = relbench.load_dataset("rel-f1")
task = relbench.load_task("rel-f1", "driver-dnf")  # will a driver DNF soon?
db = dataset.get_db()

## Build the heterogeneous graph

`make_pkey_fkey_graph` turns the relational database into a PyG `HeteroData` graph using
the foreign-key edges; a text embedder encodes string columns.

In [ ]:
data, col_stats_dict = make_pkey_fkey_graph(
    db,
    get_stype_proposal(db),
    text_embedder_cfg=TextEmbedderConfig(
        text_embedder=HashTextEmbedder(8), batch_size=None
    ),
    cache_dir=None,
)

In [ ]:
channels = 64
node_cols = {nt: data[nt].tf.col_names_dict for nt in data.node_types}
encoder = HeteroEncoder(channels, node_cols, col_stats_dict)
gnn = HeteroGraphSAGE(data.node_types, data.edge_types, channels)
head = MLP(channels, out_channels=1, num_layers=1)
optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(gnn.parameters()) + list(head.parameters()),
    lr=0.01,
)

In [ ]:
loaders = {}
for split in ["train", "val", "test"]:
    ti = get_node_train_table_input(task.get_table(split), task=task)
    loaders[split] = NeighborLoader(
        data,
        num_neighbors=[-1, -1],
        time_attr="time",
        input_nodes=ti.nodes,
        input_time=ti.time,
        transform=ti.transform,
        batch_size=256,
        shuffle=split == "train",
    )
entity = task.entity_table

## Train a few epochs

In [ ]:
def train():
    for epoch in range(1, 4):
        encoder.train(), gnn.train(), head.train()
        for batch in loaders["train"]:
            seed = batch[entity].batch_size
            x = encoder(batch.tf_dict)
            x = gnn(
                x,
                batch.edge_index_dict,
                batch.num_sampled_nodes_dict,
                batch.num_sampled_edges_dict,
            )
            pred = head(x[entity][:seed]).squeeze(-1)
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(pred, batch[entity].y.float())
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch}: train loss {loss.item():.4f}")


train()

In [ ]:
def predict():
    encoder.eval(), gnn.eval(), head.eval()
    preds = []
    for batch in loaders["test"]:
        seed = batch[entity].batch_size
        with torch.no_grad():
            x = encoder(batch.tf_dict)
            x = gnn(
                x,
                batch.edge_index_dict,
                batch.num_sampled_nodes_dict,
                batch.num_sampled_edges_dict,
            )
            preds.append(head(x[entity][:seed]).squeeze(-1).sigmoid().cpu())
    return torch.cat(preds).numpy()


test_metrics = task.evaluate(predict())
test_metrics